# 14 — Preparazione del test ufficiale (T28–T30)

Prepariamo gli stessi input per i sei modelli. **Qui non addestriamo classificatori
e non calcoliamo prestazioni sul test.** I fit sottostanti riguardano solo encoder
e scaler, che apprendono dal training. Gli esperimenti precedenti restano separati.

Percorso: protocollo → lettura → controlli → etichette → feature → preprocessing → riepilogo.

In [ ]:
from pathlib import Path
import hashlib
import json
import sys

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
assert (ROOT / 'src/data.py').is_file(), 'Avviare dalla cartella progetto o notebooks'
sys.path.insert(0, str(ROOT))
from src.data import COLUMN_NAMES, FEATURE_COLUMNS, CATEGORICAL_COLUMNS
from src.targets import ATTACK_TO_FAMILY, AUTHOR_ATTACK_TO_FAMILY, FAMILY_ORDER

OUTPUT = ROOT / 'artifacts/official_test_preparation'
OUTPUT.mkdir(parents=True, exist_ok=True)

## 1. T28 — Protocollo congelato

Usiamo i due file completi: non serve un nuovo split 80/20. Seed 1 e un fit futuro
per modello. Le configurazioni complete e le metriche sono nel JSON, prima di
osservare qualsiasi predizione. Per i dettagli: `docs/OFFICIAL_TEST_PROTOCOL.md`.

In [ ]:
protocol_path = ROOT / 'configs/official_test_v1.json'
protocol = json.loads(protocol_path.read_text(encoding='utf-8'))
protocol_sha256 = hashlib.sha256(protocol_path.read_bytes()).hexdigest()
assert protocol['features'] == list(FEATURE_COLUMNS)
assert protocol['family_order'] == list(FAMILY_ORDER)
assert protocol['label_profile'] == 'author_75b8f8b'
display(pd.DataFrame([
    {'modello': name, 'classe sklearn': item['class'], 'preprocessing': item['preprocessing']}
    for name, item in protocol['models'].items()
]))

## 2. Lettura dei CSV originali

I CSV sono nella cartella `data/raw` del progetto. Non hanno intestazione:
`header=None` conserva anche la prima riga. Verifichiamo prima l'identità dei file.

In [ ]:
for relative_path, expected_hash in protocol['input_sha256'].items():
    actual_hash = hashlib.sha256((ROOT / relative_path).read_bytes()).hexdigest()
    assert actual_hash == expected_hash, f'File diverso dal protocollo: {relative_path}'

string_types = {name: 'string' for name in (*CATEGORICAL_COLUMNS, 'attack_label')}
train = pd.read_csv(ROOT / 'data/raw/KDDTrain+.csv', header=None, names=COLUMN_NAMES, dtype=string_types)
test = pd.read_csv(ROOT / 'data/raw/KDDTest+.csv', header=None, names=COLUMN_NAMES, dtype=string_types)
display(train.head())

## 3. Controlli strutturali

Controlliamo righe, schema, valori mancanti e numeri non finiti. Non eliminiamo
righe o colonne: anche `num_outbound_cmds`, costante nel training, resta tra le 41
feature. `su_attempted=2` viene conservato, come negli esperimenti precedenti.

In [ ]:
numeric_columns = [name for name in FEATURE_COLUMNS if name not in CATEGORICAL_COLUMNS]
for name, frame in [('train', train), ('test', test)]:
    assert len(frame) == protocol['expected_rows'][name]
    assert frame.columns.tolist() == list(COLUMN_NAMES)
    assert not frame.isna().any().any(), f'Valori mancanti in {name}'
    assert np.isfinite(frame[numeric_columns + ['difficulty']].to_numpy()).all()
display(pd.DataFrame({'righe': [len(train), len(test)], 'colonne': [train.shape[1], test.shape[1]]}, index=['train', 'test']))

## 4. T29 — Etichette secondo gli autori

Una label descrive l'attacco specifico; il target del modello è la sua famiglia.
Il nuovo profilo cambia soltanto `worm` (DoS → R2L) e `httptunnel` (R2L → U2R).
Entrambi sono assenti dal training. Il mapping storico rimane disponibile e non
viene modificato. Etichette sconosciute o mancanti fermano l'esecuzione.

In [ ]:
for name, frame in [('train', train), ('test', test)]:
    unknown = frame['attack_label'].isna() | ~frame['attack_label'].isin(AUTHOR_ATTACK_TO_FAMILY)
    if unknown.any():
        raise ValueError(f"Etichette non mappate in {name}: {frame.loc[unknown, 'attack_label'].tolist()}")

y_train = train['attack_label'].map(AUTHOR_ATTACK_TO_FAMILY).rename('family')
y_test = test['attack_label'].map(AUTHOR_ATTACK_TO_FAMILY).rename('family')

In [ ]:
historical_train = train['attack_label'].map(ATTACK_TO_FAMILY)
historical_test = test['attack_label'].map(ATTACK_TO_FAMILY)
assert y_train.equals(historical_train.rename('family'))
changed_test = historical_test.ne(y_test)
assert set(test.loc[changed_test, 'attack_label']) == {'worm', 'httptunnel'}
family_counts = pd.DataFrame({
    'train': y_train.value_counts(),
    'test_storico': historical_test.value_counts(),
    'test_autori': y_test.value_counts(),
}).reindex(FAMILY_ORDER).fillna(0).astype(int)
display(family_counts)
label_audit = pd.DataFrame({
    'train': train['attack_label'].value_counts(),
    'test': test['attack_label'].value_counts(),
}).fillna(0).astype(int).sort_index()
label_audit['famiglia_storica'] = label_audit.index.map(ATTACK_TO_FAMILY)
label_audit['famiglia_autori'] = label_audit.index.map(AUTHOR_ATTACK_TO_FAMILY)
display(label_audit)
print('Righe del test con famiglia diversa:', int(changed_test.sum()))

## 5. T30 — Separazione delle feature

Selezioniamo esplicitamente le 41 colonne del dataset. Non passiamo al modello
né `attack_label`, né `difficulty`, né la famiglia derivata.

In [ ]:
X_train = train.loc[:, list(FEATURE_COLUMNS)].copy()
X_test = test.loc[:, list(FEATURE_COLUMNS)].copy()
numeric_columns = [name for name in FEATURE_COLUMNS if name not in CATEGORICAL_COLUMNS]
assert X_train.shape[1] == X_test.shape[1] == 41
assert not {'attack_label', 'difficulty', 'family'} & set(X_train.columns)

## 6. Categorie nuove, duplicati e sovrapposizioni

Un attacco nuovo non equivale a una categoria nuova di `service`, `flag` o
`protocol_type`: controlliamo separatamente i due aspetti. I duplicati e le
sovrapposizioni sono descritti, ma non rimossi per preservare il protocollo.

In [ ]:
unseen_categories = {
    column: sorted(set(X_test[column]) - set(X_train[column]))
    for column in CATEGORICAL_COLUMNS
}
unseen_attacks = sorted(set(test['attack_label']) - set(train['attack_label']))
constant_features = X_train.columns[X_train.nunique() == 1].tolist()
overlap_features = len(X_train.drop_duplicates().merge(X_test.drop_duplicates(), on=list(FEATURE_COLUMNS)))
data_checks = {
    'unseen_categories': unseen_categories,
    'unseen_attacks': unseen_attacks,
    'constant_train_features': constant_features,
    'duplicate_feature_rows_train': int(X_train.duplicated().sum()),
    'duplicate_feature_rows_test': int(X_test.duplicated().sum()),
    'shared_unique_feature_vectors': overlap_features,
}
display(data_checks)

## 7. Preprocessing appreso soltanto sul training

Due configurazioni: alberi con numeriche originali; gli altri modelli con
StandardScaler. In entrambi i casi le tre categoriche diventano colonne one-hot.
`fit_transform` vede solo X_train; `transform` applica al test ciò che è già stato
appreso. Questi fit preparano i dati: non sono addestramenti dei sei modelli.

In [ ]:
tree_preprocessor = ColumnTransformer([
    ('categorical', OneHotEncoder(handle_unknown='ignore', sparse_output=False), list(CATEGORICAL_COLUMNS)),
    ('numeric', 'passthrough', numeric_columns),
])
scaled_preprocessor = ColumnTransformer([
    ('categorical', OneHotEncoder(handle_unknown='ignore', sparse_output=False), list(CATEGORICAL_COLUMNS)),
    ('numeric', StandardScaler(), numeric_columns),
])

X_train_tree = tree_preprocessor.fit_transform(X_train)
X_test_tree = tree_preprocessor.transform(X_test)
X_train_scaled = scaled_preprocessor.fit_transform(X_train)
X_test_scaled = scaled_preprocessor.transform(X_test)

In [ ]:
scaler = scaled_preprocessor.named_transformers_['numeric']
np.testing.assert_allclose(scaler.mean_, X_train[numeric_columns].mean().to_numpy())
np.testing.assert_allclose(scaler.var_, X_train[numeric_columns].var(ddof=0).to_numpy())
assert scaler.n_samples_seen_ == len(X_train)
for preprocessor in [tree_preprocessor, scaled_preprocessor]:
    encoder = preprocessor.named_transformers_['categorical']
    for column, categories in zip(CATEGORICAL_COLUMNS, encoder.categories_):
        assert set(categories) == set(X_train[column])
for matrix in [X_train_tree, X_test_tree, X_train_scaled, X_test_scaled]:
    assert np.isfinite(matrix).all()
assert X_train_tree.shape[1] == X_test_tree.shape[1] == X_train_scaled.shape[1] == X_test_scaled.shape[1]
display(pd.DataFrame({
    'train': X_train_tree.shape,
    'test': X_test_tree.shape,
}, index=['righe', 'feature dopo one-hot']))

## 8. Evidenze salvate e prossimo passo

Salviamo conteggi e controlli in una cartella nuova, senza sovrascrivere risultati
storici. Non salviamo matrici trasformate gigantesche: ogni notebook di modello
potrà ricostruirle sul training con queste stesse istruzioni.

Prossimo ticket: **T31, Decision Tree con profondità 15** sul training completo,
poi valutazione unica sul test ufficiale. Non viene eseguito qui.

In [ ]:
family_counts.to_csv(OUTPUT / 'family_counts.csv', index_label='family')
label_audit.to_csv(OUTPUT / 'label_audit.csv', index_label='attack_label')
audit = {
    'protocol_id': protocol['protocol_id'],
    'protocol_sha256': protocol_sha256,
    'input_sha256': protocol['input_sha256'],
    'label_profile': protocol['label_profile'],
    'rows': {'train': len(train), 'test': len(test)},
    'changed_training_labels': int(historical_train.ne(y_train).sum()),
    'changed_test_labels': int(changed_test.sum()),
    'feature_count_raw': 41,
    'feature_count_encoded': X_train_tree.shape[1],
    'preprocessing_fitted_on': 'KDDTrain+ only',
    'scaler_samples_seen': int(scaler.n_samples_seen_),
    'scaler_mean': scaler.mean_.tolist(),
    'scaler_variance': scaler.var_.tolist(),
    'encoder_categories': [x.tolist() for x in scaled_preprocessor.named_transformers_['categorical'].categories_],
    'model_fits': 0,
    'official_test_predictions': 0,
    **data_checks,
}
(OUTPUT / 'audit.json').write_text(json.dumps(audit, indent=2, ensure_ascii=False) + '\n', encoding='utf-8')
print('T28–T30: controlli completati; nessun modello addestrato.')